
# 03 — TF-IDF-Based Movie Recommender

## Objective
Build a personalized movie recommendation system using movie overviews,
TF-IDF vectorization, and cosine similarity.

## Recommendation workflow
1. Preprocess movie descriptions.
2. Convert descriptions into numerical vectors using TF-IDF.
3. Calculate cosine similarity between movies.
4. Use the user's ratings of watched movies to build a preference profile.
5. Rank unseen movies and recommend the top K.

## Dataset
movies_clean.csv


In [72]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split

In [73]:

movies_clean = pd.read_csv("/content/movies_clean.csv")

print("Dataset shape:", movies_clean.shape)
print("\nColumns:")
print(movies_clean.columns.tolist())

display(movies_clean.head())


Dataset shape: (12794, 8)

Columns:
['genres', 'overview', 'release_date', 'title', 'release_year', 'tmdbId', 'avg_rating', 'rating_count']


,genres,overview,release_date,title,release_year,tmdbId,avg_rating,rating_count
0,"[{'id': 18, 'name': 'Drama'}, {'id': 10749, 'n...",An atmospheric coming-of-age story featuring a...,2010-05-25,Venice,2010,79782,2.977612,67.0
1,"[{'id': 35, 'name': 'Comedy'}, {'id': 27, 'nam...","The town of Derry has a secret, but no one tol...",2013-10-12,The Sleepover,2013,141210,2.953488,43.0
2,"[{'id': 18, 'name': 'Drama'}]","As her surroundings are invaded by outsiders, ...",2012-06-20,The Farmer's Wife,2012,143750,2.891304,46.0
3,"[{'id': 99, 'name': 'Documentary'}]","Using personal stories, this powerful document...",2012-03-22,A Place at the Table,2012,84198,2.862500,80.0
4,"[{'id': 10749, 'name': 'Romance'}, {'id': 18, ...",She's as hot as Britney Spears. Hotter. She pa...,2012-10-09,Romance,2012,171982,2.789007,564.0


In [74]:
# Check for missing descriptions
print("Missing overviews:", movies_clean["overview"].isna().sum())

# Check for empty descriptions
empty_overviews = (
    movies_clean["overview"]
    .fillna("")
    .str.strip()
    .eq("")
    .sum()
)

print("Empty overviews:", empty_overviews)

# Display a few examples
for i in range(5):
    print(f"\nMovie: {movies_clean.loc[i, 'title']}")
    print(f"Overview: {movies_clean.loc[i, 'overview']}")

Missing overviews: 296
Empty overviews: 296

Movie: Venice
Overview: An atmospheric coming-of-age story featuring an imaginative young boy named Marek who dreams of escaping an increasingly dangerous Poland on the eve of war for beautiful Venice.

Movie: The Sleepover
Overview: The town of Derry has a secret, but no one told the new kid. It's gonna be a long night.

Movie: The Farmer's Wife
Overview: As her surroundings are invaded by outsiders, THE FARMER'S WIFE stoically carries out her daily tasks in preparation for what will be her final day on her farm, the only land she has known. Unable to accept this forced future, subconscious memories return that ultimately lead her to connect more to the countryside of her youth than ever before.

Movie: A Place at the Table
Overview: Using personal stories, this powerful documentary illuminates the plight of the 49 million Americans struggling with food insecurity. A single mother, a small-town policeman and a farmer are among those for who

In [75]:
# Replace missing descriptions with empty strings
movies_clean["overview_clean"] = (
    movies_clean["overview"]
    .fillna("")
    .str.strip()
)

# Check the result
print("Missing values:", movies_clean["overview_clean"].isna().sum())

print(
    "Empty descriptions:",
    movies_clean["overview_clean"].eq("").sum()
)

print("\nSample cleaned descriptions:")
display(movies_clean[["title", "overview_clean"]].head())

Missing values: 0
Empty descriptions: 296

Sample cleaned descriptions:


,title,overview_clean
0,Venice,An atmospheric coming-of-age story featuring a...
1,The Sleepover,"The town of Derry has a secret, but no one tol..."
2,The Farmer's Wife,"As her surroundings are invaded by outsiders, ..."
3,A Place at the Table,"Using personal stories, this powerful document..."
4,Romance,She's as hot as Britney Spears. Hotter. She pa...


In [76]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    stop_words="english",
    max_features=20000,
    ngram_range=(1, 2),
    min_df=2
)


In [77]:
# Convert movie descriptions into TF-IDF feature vectors
tfidf_matrix = tfidf.fit_transform(
    movies_clean["overview_clean"]
)

print("TF-IDF matrix created successfully!")
print("Matrix shape:", tfidf_matrix.shape)
print("Number of movies:", tfidf_matrix.shape[0])
print("Number of features:", tfidf_matrix.shape[1])

TF-IDF matrix created successfully!
Matrix shape: (12794, 20000)
Number of movies: 12794
Number of features: 20000


In [78]:

# Calculate similarity between every pair of movies
cosine_sim = cosine_similarity(tfidf_matrix, dense_output=False)

print("Cosine similarity matrix created!")
print("Matrix shape:", cosine_sim.shape)

Cosine similarity matrix created!
Matrix shape: (12794, 12794)


In [79]:
# Map each TMDB movie ID to its row position
movie_id_to_index = pd.Series(
    movies_clean.index,
    index=movies_clean["tmdbId"]
).to_dict()

print("Movie ID mapping created.")
print("Number of mapped movies:", len(movie_id_to_index))
print("Example mapping:", list(movie_id_to_index.items())[:5])

Movie ID mapping created.
Number of mapped movies: 12794
Example mapping: [(79782, 0), (141210, 1), (143750, 2), (84198, 3), (171982, 4)]


In [105]:

def recommend_personalized_movies(movie_ratings, k=10):
    """
    Recommend movies using TF-IDF and a weighted user preference profile.

    Parameters
    ----------
    movie_ratings : dict
        {tmdbId: user_rating}, where ratings range from 0.5 to 5.0
        in increments of 0.5.
    k : int
        Number of recommendations to return.

    Returns
    -------
    pandas.DataFrame
        Top-k recommended movies, excluding movies already rated
        and movies without overviews.
    """

    # 1. Validate inputs
    if not isinstance(movie_ratings, dict) or not movie_ratings:
        raise ValueError("Please provide ratings for at least one movie.")

    if not isinstance(k, (int, np.integer)) or isinstance(k, (bool, np.bool_)) or k <= 0:
        raise ValueError("k must be a positive integer.")

    # 2. Validate movie IDs and ratings
    for movie_id, rating in movie_ratings.items():

        if movie_id not in movie_id_to_index:
            raise ValueError(
                f"Movie ID {movie_id} is not in our catalogue."
            )

        if (
            not isinstance(rating, (int, float, np.integer, np.floating))
            or isinstance(rating, (bool, np.bool_))
            or not np.isfinite(rating)
            or not 0.5 <= rating <= 5.0
            or (rating * 2) % 1 != 0
        ):
            raise ValueError(
                f"Rating for movie {movie_id} must be between "
                "0.5 and 5.0 in increments of 0.5."
            )

    # 3. Build the user's weighted preference profile
    profile = None

    for movie_id, rating in movie_ratings.items():
        movie_index = movie_id_to_index[movie_id]
        movie_vector = tfidf_matrix[movie_index]

        # Ratings above 3 receive positive weights.
        # Ratings below 3 receive negative weights.
        # Neutral ratings receive a small positive weight.
        weight = 0.1 if rating == 3 else rating - 3

        if profile is None:
            profile = weight * movie_vector
        else:
            profile = profile + weight * movie_vector

    # 4. Check whether the profile is informative
    profile_norm = np.sqrt(profile.multiply(profile).sum())

    if profile_norm < 1e-10:
        raise ValueError(
            "Your ratings produced an uninformative profile. "
            "Please rate some different movies."
        )

    # 5. Calculate cosine similarity with all movie descriptions
    scores = cosine_similarity(profile, tfidf_matrix).ravel()

    # 6. Exclude movies already rated and movies without overviews
    watched_indices = {
        movie_id_to_index[movie_id]
        for movie_id in movie_ratings
    }

    has_overview = movies_clean["overview_clean"].ne("").to_numpy()
    valid_indices = np.flatnonzero(has_overview)

    candidate_indices = [
        idx for idx in valid_indices
        if idx not in watched_indices
    ]

    # 7. Rank candidates by similarity score
    ranked_indices = sorted(
        candidate_indices,
        key=lambda idx: scores[idx],
        reverse=True
    )[:k]

    # 8. Return recommendations with their details
    recommendations = movies_clean.iloc[ranked_indices].copy()
    recommendations["content_score"] = scores[ranked_indices]

    return recommendations[
        [
            "tmdbId",
            "title",
            "release_year",
            "genres",
            "overview",
            "avg_rating",
            "rating_count",
            "content_score"
        ]
    ].reset_index(drop=True)


In [81]:
# Display movies available in our catalogue
display(
    movies_clean[
        ["tmdbId", "title", "release_year", "avg_rating"]
    ].sample(15, random_state=42).reset_index(drop=True)
)

,tmdbId,title,release_year,avg_rating
0,203834,Jessabelle,2014,2.971014
1,98344,Warriors of the Rainbow: Seediq Bale - Part 1:...,2011,3.812500
2,77948,The Cold Light of Day,2012,2.528169
3,332177,Vendetta,2015,2.800000
4,82469,The Kick,2011,3.000000
5,351863,The Shaman,2015,3.250000
6,257109,Bite Size,2014,2.000000
7,66111,Luster,2010,2.846154
8,73941,Limelight,2011,3.500000
9,85819,Some Days Are Better Than Others,2011,3.000000


In [82]:
# Simulated ratings from one user
sample_user_ratings = {
    98344: 5.0,
    351863: 4.5,
    73941: 3.0,
    257109: 1.5,
    315669: 2.0
}

# Generate 10 personalized recommendations
recommendations = recommend_personalized_movies(
    sample_user_ratings,
    k=10
)

display(
    recommendations[
        ["title", "release_year", "avg_rating", "content_score"]
    ]
)

,title,release_year,avg_rating,content_score
0,Prologue,2015,3.600000,0.142084
1,Alien vs. Ninja,2010,2.300000,0.119734
2,47 Ronin,2013,3.064910,0.114269
3,Tumult,2012,3.000000,0.108838
4,Hacksaw Ridge,2016,3.872426,0.095818
5,The Great Martian War 1913 - 1917,2013,3.166667,0.093236
6,The Ninja Immovable Heart,2014,2.000000,0.091375
7,Jack the Giant Slayer,2013,3.010703,0.090884
8,Tears of Steel,2012,2.818182,0.089090
9,Raze,2013,2.347826,0.088873


In [83]:
recommendations.columns

Index(['tmdbId', 'title', 'release_year', 'genres', 'overview', 'avg_rating',
       'rating_count', 'content_score'],
      dtype='object')

In [84]:
# Print complete descriptions of movies rated by the user

print("MOVIES RATED BY THE USER")


for movie_id, rating in sample_user_ratings.items():
    movie = movies_clean.loc[
        movies_clean["tmdbId"] == movie_id
    ].iloc[0]

    print(f"\nTitle: {movie['title']}")
    print(f"User rating: {rating}/5")
    print(f"Overview: {movie['overview_clean']}")
    print("-" * 80)


# Print complete descriptions of recommended movies
print("\n\n")
print("RECOMMENDED MOVIES")


for _, movie in recommendations.iterrows():
    print(f"\nTitle: {movie['title']}")
    print(f"Content score: {movie['content_score']:.4f}")
    print(f"Overview: {movie['overview']}")
    print("-" * 80)

MOVIES RATED BY THE USER

Title: Warriors of the Rainbow: Seediq Bale - Part 1: The Sun Flag
User rating: 5.0/5
Overview: Mouna Rudo was born and raised among the Seediq people, an indigenous tribe in Taiwan, and as he grew to be a man he became a member of the Seediq Bale, a courageous band of native warriors. However, Rudo's way of life is threatened under the yoke of occupying forces from Japan, who took over the nation in 1895. As Rudo sees the traditions and honor of his people stripped away, he realizes the time has come to fight back, and in 1930 he brings together a group of former Seediq Bale soldiers, many of whom have been reduced to infighting, and molds them into a revolutionary army. Rudo and his comrades make their stand when they confront Japanese occupation troops at a youth athletic event, leading to a violent confrontation between the Seediq forces and their oppressors. Warriors of the Rainbow: Seediq Bale - Part 1: The Sun Flag is Part one of the two-part, four-hour

# Model Evaluation

## Model Evaluation Strategy

We will evaluate our TF-IDF-based movie recommendation system using a **train-test split of individual user ratings**. First, we will map MovieLens movie IDs to TMDB IDs and retain only ratings corresponding to movies in our recommendation catalogue (movies released in 2010 or later). To ensure that users have sufficient rating history, we will initially consider users who have rated at least 100 movies in this catalogue.

For each eligible user, we will split their ratings into training and test sets, using 80% for training and 20% for testing. The training ratings will be used to construct the user's preference profile, which combines the TF-IDF vectors of rated movies according to their ratings. The system will then recommend the top 10 unseen movies using cosine similarity.

We will evaluate these recommendations against the user's held-out test ratings, treating movies rated **4 or higher out of 5** as relevant. We will use **Precision@10, Recall@10, and NDCG@10** to measure how effectively the system retrieves relevant movies and ranks them near the top. We will ensure that test ratings are not used to construct user profiles or generate recommendations.

This evaluation will establish a baseline for our content-based recommender. Later, we can compare its performance with our popularity-based and genre-based recommenders.


In [85]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [86]:
import os

movie_match_dir = "/content/drive/MyDrive/MovieMatch"

print("Folder exists:", os.path.exists(movie_match_dir))
print("\nFiles in MovieMatch folder:")

for file in os.listdir(movie_match_dir):
    print(file)

Folder exists: True

Files in MovieMatch folder:
links.csv
movies_metadata.csv
ratings.csv


In [87]:

movie_match_dir = "/content/drive/MyDrive/MovieMatch"

# Read only the column names
ratings_columns = pd.read_csv(
    os.path.join(movie_match_dir, "ratings.csv"),
    nrows=5
)

links_columns = pd.read_csv(
    os.path.join(movie_match_dir, "links.csv"),
    nrows=5
)

print("RATINGS.CSV")
print("Columns:", ratings_columns.columns.tolist())
display(ratings_columns.head())

print("\nLINKS.CSV")
print("Columns:", links_columns.columns.tolist())
display(links_columns.head())

RATINGS.CSV
Columns: ['userId', 'movieId', 'rating', 'timestamp']


,userId,movieId,rating,timestamp
0,1,110,1.0,1425941529
1,1,147,4.5,1425942435
2,1,858,5.0,1425941523
3,1,1221,5.0,1425941546
4,1,1246,5.0,1425941556



LINKS.CSV
Columns: ['movieId', 'imdbId', 'tmdbId']


,movieId,imdbId,tmdbId
0,1,114709,862
1,2,113497,8844
2,3,113228,15602
3,4,114885,31357
4,5,113041,11862


In [88]:
movie_match_dir = "/content/drive/MyDrive/MovieMatch"

# Load only the columns needed for evaluation
ratings = pd.read_csv(
    os.path.join(movie_match_dir, "ratings.csv"),
    usecols=["userId", "movieId", "rating"]
)

links = pd.read_csv(
    os.path.join(movie_match_dir, "links.csv"),
    usecols=["movieId", "tmdbId"]
)

# Remove rows without a valid TMDB ID
links = links.dropna(subset=["tmdbId"]).copy()

# Convert TMDB IDs to integers
links["tmdbId"] = links["tmdbId"].astype(int)

# Map MovieLens movie IDs to TMDB IDs
ratings = ratings.merge(
    links,
    on="movieId",
    how="inner"
)

print("Ratings after ID mapping:", ratings.shape)
print("Unique users:", ratings["userId"].nunique())
print("Unique MovieLens movies:", ratings["movieId"].nunique())
print("Unique TMDB movies:", ratings["tmdbId"].nunique())

display(ratings.head())

Ratings after ID mapping: (26010786, 4)
Unique users: 270888
Unique MovieLens movies: 44902
Unique TMDB movies: 44872


,userId,movieId,rating,tmdbId
0,1,110,1.0,197
1,1,147,4.5,10474
2,1,858,5.0,238
3,1,1221,5.0,240
4,1,1246,5.0,207


In [89]:
# Get the TMDB IDs of movies in our recommendation catalogue
eligible_tmdb_ids = set(movies_clean["tmdbId"])

# Keep only ratings belonging to eligible movies
ratings_eligible = ratings[
    ratings["tmdbId"].isin(eligible_tmdb_ids)
].copy()

print("Ratings before catalogue filtering:", len(ratings))
print("Ratings after catalogue filtering:", len(ratings_eligible))
print("Unique users:", ratings_eligible["userId"].nunique())
print("Unique movies:", ratings_eligible["tmdbId"].nunique())

display(ratings_eligible.head())

Ratings before catalogue filtering: 26010786
Ratings after catalogue filtering: 1913317
Unique users: 67167
Unique movies: 12652


,userId,movieId,rating,tmdbId
19,1,81834,5.0,12444
20,1,91500,2.5,70160
21,1,91542,5.0,58574
22,1,92439,5.0,64678
23,1,96821,5.0,84892


In [90]:
# Count how many eligible movies each user has rated
user_rating_counts = (
    ratings_eligible.groupby("userId")["tmdbId"]
    .nunique()
    .sort_values(ascending=False)
)

print("Total users:", len(user_rating_counts))
print("\nRating-count distribution:")
display(user_rating_counts.describe())

print("\nUsers with at least 100 eligible ratings:",
      (user_rating_counts >= 100).sum())

print("Users with at least 200 eligible ratings:",
      (user_rating_counts >= 200).sum())

Total users: 67167

Rating-count distribution:


,tmdbId
count,67167.000000
mean,28.485834
std,66.496096
min,1.000000
25%,3.000000
50%,8.000000
75%,26.000000
max,5294.000000



Users with at least 100 eligible ratings: 4582
Users with at least 200 eligible ratings: 1540


In [91]:
# Identify users with at least 100 distinct eligible movies rated
eligible_users = user_rating_counts[
    user_rating_counts >= 100
].index

# Keep ratings for those users
ratings_filtered = ratings_eligible[
    ratings_eligible["userId"].isin(eligible_users)
].copy()

print("Eligible users:", len(eligible_users))
print("Ratings retained:", len(ratings_filtered))
print("Unique users retained:", ratings_filtered["userId"].nunique())
print("Unique movies retained:", ratings_filtered["tmdbId"].nunique())

# Verify the minimum number of distinct movies per user
filtered_counts = ratings_filtered.groupby("userId")["tmdbId"].nunique()

print("\nMinimum ratings per retained user:", filtered_counts.min())
print("Median ratings per retained user:", filtered_counts.median())

Eligible users: 4582
Ratings retained: 946388
Unique users retained: 4582
Unique movies retained: 12208

Minimum ratings per retained user: 100
Median ratings per retained user: 159.0


In [92]:
RANDOM_STATE = 42

train_parts = []
test_parts = []

for user_id, user_data in ratings_filtered.groupby("userId"):
    user_train, user_test = train_test_split(
        user_data,
        test_size=0.20,
        random_state=RANDOM_STATE
    )

    train_parts.append(user_train)
    test_parts.append(user_test)

ratings_train = pd.concat(train_parts, ignore_index=True)
ratings_test = pd.concat(test_parts, ignore_index=True)

print("Training ratings:", len(ratings_train))
print("Test ratings:", len(ratings_test))

print("\nUsers in training:", ratings_train["userId"].nunique())
print("Users in testing:", ratings_test["userId"].nunique())

print("\nTraining fraction:",
      round(len(ratings_train) / len(ratings_filtered), 3))
print("Test fraction:",
      round(len(ratings_test) / len(ratings_filtered), 3))

Training ratings: 755290
Test ratings: 191098

Users in training: 4582
Users in testing: 4582

Training fraction: 0.798
Test fraction: 0.202


In [93]:
# Check for overlapping user-movie pairs
train_pairs = set(
    zip(ratings_train["userId"], ratings_train["tmdbId"])
)

test_pairs = set(
    zip(ratings_test["userId"], ratings_test["tmdbId"])
)

overlap = train_pairs.intersection(test_pairs)

print("Overlapping user-movie pairs:", len(overlap))

# Verify user coverage
train_users = set(ratings_train["userId"])
test_users = set(ratings_test["userId"])

print("Users missing from training:", len(test_users - train_users))
print("Users missing from testing:", len(train_users - test_users))

Overlapping user-movie pairs: 1
Users missing from training: 0
Users missing from testing: 0


In [94]:
# Find overlapping user-movie pairs
overlap_df = ratings_train.merge(
    ratings_test,
    on=["userId", "tmdbId"],
    suffixes=("_train", "_test")
)

display(overlap_df)

,userId,movieId_train,rating_train,tmdbId,movieId_test,rating_test
0,222647,114342,3.5,265189,175813,3.5


In [95]:
# Identify user-TMDB pairs already present in training
train_pairs = set(
    zip(ratings_train["userId"], ratings_train["tmdbId"])
)

# Mark test rows whose user-TMDB pair is already in training
test_pairs = list(
    zip(ratings_test["userId"], ratings_test["tmdbId"])
)

overlap_mask = [
    pair in train_pairs for pair in test_pairs
]

print("Overlapping test rows to remove:", sum(overlap_mask))

# Remove the overlapping row(s) from the test set
ratings_test = ratings_test.loc[
    [not is_overlap for is_overlap in overlap_mask]
].reset_index(drop=True)

# Verify again
train_pairs = set(
    zip(ratings_train["userId"], ratings_train["tmdbId"])
)
test_pairs = set(
    zip(ratings_test["userId"], ratings_test["tmdbId"])
)

print("Overlapping pairs after fix:", len(train_pairs.intersection(test_pairs)))
print("Training ratings:", len(ratings_train))
print("Test ratings after fix:", len(ratings_test))

Overlapping test rows to remove: 1
Overlapping pairs after fix: 0
Training ratings: 755290
Test ratings after fix: 191097


In [96]:

# Define the relevance threshold
RELEVANCE_THRESHOLD = 3.5

# Mark test ratings as relevant or non-relevant
ratings_test = ratings_test.copy()

ratings_test["relevant"] = (
    ratings_test["rating"] >= RELEVANCE_THRESHOLD
)

print("Total test ratings:", len(ratings_test))
print("Relevant test ratings:",
      ratings_test["relevant"].sum())
print("Non-relevant test ratings:",
      (~ratings_test["relevant"]).sum())

print("\nTest rating distribution:")
display(ratings_test["rating"].value_counts().sort_index())


Total test ratings: 191097
Relevant test ratings: 112307
Non-relevant test ratings: 78790

Test rating distribution:


,count
rating,
0.5,5346
1.0,4520
1.5,4733
2.0,11755
2.5,17056
3.0,35380
3.5,39683
4.0,41716
4.5,17452


In [97]:

# Count relevant test movies for each user
relevant_counts = (
    ratings_test.groupby("userId")["relevant"].sum()
)

users_with_relevant = relevant_counts[relevant_counts > 0]
users_without_relevant = relevant_counts[relevant_counts == 0]

print("Total users in test set:", len(relevant_counts))
print("Users with at least one relevant test movie:",
      len(users_with_relevant))
print("Users with no relevant test movies:",
      len(users_without_relevant))

print("\nRelevant test movies per user:")
display(relevant_counts.describe())


Total users in test set: 4582
Users with at least one relevant test movie: 4574
Users with no relevant test movies: 8

Relevant test movies per user:


,relevant
count,4582.000000
mean,24.510476
std,16.573459
min,0.000000
25%,15.000000
50%,21.000000
75%,29.000000
max,448.000000


In [98]:

# Evaluate users who have at least one relevant test movie
evaluation_users = users_with_relevant.index.tolist()

# Verify training history for every evaluation user
train_user_counts = ratings_train.groupby("userId")["tmdbId"].nunique()

users_without_training = [
    user_id
    for user_id in evaluation_users
    if user_id not in train_user_counts.index
]

print("Users selected for evaluation:", len(evaluation_users))
print("Users without training history:", len(users_without_training))

print("\nTraining movies per evaluation user:")
display(train_user_counts.loc[evaluation_users].describe())


Users selected for evaluation: 4574
Users without training history: 0

Training movies per evaluation user:


,tmdbId
count,4574.000000
mean,164.888282
std,127.297228
min,80.000000
25%,98.000000
50%,127.000000
75%,185.000000
max,4234.000000


In [99]:

# Build each user's training-rating dictionary
train_ratings_by_user = {}

for user_id, group in ratings_train.groupby("userId"):
    train_ratings_by_user[user_id] = dict(
        zip(group["tmdbId"], group["rating"])
    )

# Build each user's set of relevant test movies
relevant_test_by_user = {}

for user_id, group in ratings_test[
    ratings_test["relevant"]
].groupby("userId"):
    relevant_test_by_user[user_id] = set(group["tmdbId"])

print("Users with training profiles:", len(train_ratings_by_user))
print("Users with relevant test movies:", len(relevant_test_by_user))

# Inspect one user without evaluating recommendations yet
example_user = evaluation_users[0]

print("\nExample user:", example_user)
print("Training movies:", len(train_ratings_by_user[example_user]))
print("Relevant test movies:", len(relevant_test_by_user[example_user]))


Users with training profiles: 4582
Users with relevant test movies: 4574

Example user: 231
Training movies: 248
Relevant test movies: 24


In [100]:

def evaluate_recommendations(recommended_ids, relevant_ids, k=10):
    # Keep only the top-k recommendations
    recommended_ids = recommended_ids[:k]

    # Convert recommendations into binary relevance labels
    relevance = np.array([
        1 if movie_id in relevant_ids else 0
        for movie_id in recommended_ids
    ])

    # Precision@K
    precision = relevance.sum() / k

    # Recall@K
    recall = relevance.sum() / len(relevant_ids) if relevant_ids else 0.0

    # DCG@K: reward relevant movies more when they rank higher
    discounts = np.log2(np.arange(2, len(relevance) + 2))
    dcg = np.sum(relevance / discounts)

    # IDCG@K: score of the ideal ranking
    ideal_relevance = np.ones(min(len(relevant_ids), k))
    ideal_discounts = np.log2(np.arange(2, len(ideal_relevance) + 2))
    idcg = np.sum(ideal_relevance / ideal_discounts)

    # NDCG@K
    ndcg = dcg / idcg if idcg > 0 else 0.0

    return {
        f"Precision@{k}": precision,
        f"Recall@{k}": recall,
        f"NDCG@{k}": ndcg
    }


In [106]:
from tqdm.auto import tqdm

K = 10

evaluation_results = []

for user_id in tqdm(evaluation_users, desc="Evaluating users"):

    # Get this user's training ratings and relevant test movies
    user_train_ratings = train_ratings_by_user[user_id]
    user_relevant_test = relevant_test_by_user[user_id]

    # Generate top-10 recommendations using our existing function
    recommendations = recommend_personalized_movies(
        user_train_ratings,
        k=K
    )

    recommended_ids = recommendations["tmdbId"].tolist()

    # Calculate evaluation metrics for this user
    scores = evaluate_recommendations(
        recommended_ids,
        user_relevant_test,
        k=K
    )

    # Store the results
    evaluation_results.append({
        "userId": user_id,
        **scores
    })

# Convert results into a DataFrame
evaluation_results = pd.DataFrame(evaluation_results)

print("Users evaluated:", len(evaluation_results))
print("\nFirst five user-level results:")
display(evaluation_results.head())


Evaluating users:   0%|          | 0/4574 [00:00<?, ?it/s]

Users evaluated: 4574

First five user-level results:


,userId,Precision@10,Recall@10,NDCG@10
0,231,0.0,0.000000,0.000000
1,242,0.1,0.052632,0.069431
2,332,0.0,0.000000,0.000000
3,429,0.0,0.000000,0.000000
4,457,0.0,0.000000,0.000000


In [107]:

# Calculate mean evaluation metrics across all evaluated users
mean_metrics = evaluation_results[
    ["Precision@10", "Recall@10", "NDCG@10"]
].mean()

print("Mean evaluation metrics:")
display(mean_metrics.to_frame(name="Mean score"))

# Examine recall across users
print("\nRecall@10 distribution:")
display(evaluation_results["Recall@10"].describe())

# Count users whose recall is zero or positive
zero_recall_users = (evaluation_results["Recall@10"] == 0).sum()
positive_recall_users = (evaluation_results["Recall@10"] > 0).sum()

print("\nUsers with zero Recall@10:", zero_recall_users)
print("Users with positive Recall@10:", positive_recall_users)
print(
    "Percentage of users with zero recall:",
    round(zero_recall_users / len(evaluation_results) * 100, 2),
    "%"
)


Mean evaluation metrics:


,Mean score
Precision@10,0.020398
Recall@10,0.009614
NDCG@10,0.022633



Recall@10 distribution:


,Recall@10
count,4574.000000
mean,0.009614
std,0.026773
min,0.000000
25%,0.000000
50%,0.000000
75%,0.000000
max,0.400000



Users with zero Recall@10: 3783
Users with positive Recall@10: 791
Percentage of users with zero recall: 82.71 %


### Limitations of the Content-Based Recommendation System

Our TF-IDF-based recommendation system identifies movies whose overviews are textually similar to the content associated with a user's rating history. However, textual similarity does not necessarily imply that a user will enjoy a recommended movie. Two movies may have similar descriptions but differ in execution, tone, characters, or other factors that influence personal preference.

Furthermore, the evaluation uses historical ratings as an imperfect proxy for user preferences. A movie rated positively in the test set may not be recommended if its overview differs from the user's learned content profile. Conversely, a recommended movie may be relevant to the user's interests even if it does not appear among their positively rated test movies. Missing ratings, incomplete viewing histories, and missing movie descriptions can also affect the measured results.IF the user havent ever seen the movie movie recommended by the model and hence not rated it would not mean that the recommendation was incorrect. Also Low recall score can mean that user can explore new movie that he/she haven't seen yet

Therefore, Precision@10, Recall@10, and NDCG@10 measure how well the system retrieves held-out positively rated movies, rather than guaranteeing whether a user will enjoy a recommendation. These limitations motivate future improvements, such as incorporating genre information or collaborative filtering alongside textual similarity.

In [108]:

# Save individual user-level evaluation results
evaluation_results.to_csv(
    "tfidf_evaluation_results.csv",
    index=False
)

# Save mean evaluation metrics
mean_metrics.to_frame(name="Mean score").to_csv(
    "tfidf_evaluation_summary.csv"
)

print("Saved user-level results: tfidf_evaluation_results.csv")
print("Saved summary metrics: tfidf_evaluation_summary.csv")


Saved user-level results: tfidf_evaluation_results.csv
Saved summary metrics: tfidf_evaluation_summary.csv


In [110]:

from pathlib import Path
import joblib

# Folder for MovieMatch backend artifacts
ARTIFACT_DIR = Path("/content/drive/MyDrive/MovieMatch/backend_artifacts")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

# 1. Save the fitted TF-IDF vectorizer
joblib.dump(
    tfidf,
    ARTIFACT_DIR / "tfidf_vectorizer.pkl"
)

# 2. Save the sparse TF-IDF movie matrix
joblib.dump(
    tfidf_matrix,
    ARTIFACT_DIR / "tfidf_matrix.pkl",
    compress=3
)

# 3. Save the TMDB ID -> TF-IDF row-index mapping
joblib.dump(
    movie_id_to_index,
    ARTIFACT_DIR / "movie_id_to_index.pkl"
)

# 4. Save the movie catalogue used by the recommender
movies_clean.to_csv(
    ARTIFACT_DIR / "movies_clean.csv",
    index=False
)

# 5. Filter MovieLens-TMDB links to our post-2010 catalogue
catalogue_tmdb_ids = set(movies_clean["tmdbId"].dropna().astype(int))

links_columns = links[["movieId", "tmdbId"]].copy()
links_columns = links_columns.dropna(subset=["tmdbId"])
links_columns["tmdbId"] = links_columns["tmdbId"].astype(int)

links_columns = links_columns[
    links_columns["tmdbId"].isin(catalogue_tmdb_ids)
].drop_duplicates()

links_columns.to_csv(
    ARTIFACT_DIR / "links_columns.csv",
    index=False
)

print("Backend artifacts saved to:", ARTIFACT_DIR)
print("\nFiles saved:")
for file_path in sorted(ARTIFACT_DIR.iterdir()):
    print(f"{file_path.name}: {file_path.stat().st_size / (1024**2):.2f} MB")

print("\nFiltered links rows:", len(links_columns))
print("Unique MovieLens movie IDs:", links_columns["movieId"].nunique())
print("Unique TMDB movie IDs:", links_columns["tmdbId"].nunique())


Backend artifacts saved to: /content/drive/MyDrive/MovieMatch/backend_artifacts

Files saved:
links_columns.csv: 0.17 MB
movie_id_to_index.pkl: 0.09 MB
movies_clean.csv: 9.50 MB
tfidf_matrix.pkl: 3.22 MB
tfidf_vectorizer.pkl: 0.74 MB

Filtered links rows: 12800
Unique MovieLens movie IDs: 12800
Unique TMDB movie IDs: 12794
